### Transform Races Data

1. Read bronze_races table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (raceName → race_name, circuitId → circuit_id)
4. Rename columns to make them more meaningful (date → race_date)
5. Remove duplicate records
6. Transform values of column race_name to Title Case
7. Write the transformed data to silver_races table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.races"
Silver_table = F"{catalog_name}.{silver_schema}.races"


In [0]:
races_df = spark.read.table(bronze_table)


In [0]:
display(races_df)

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("timestamp"),
    F.col("source_file")
)




In [0]:
display(races_selected_df)

In [0]:
races_remaned_df = (
    races_selected_df
        .withColumnsRenamed({
            "raceName": "race_name",
            "circuitId": "circuit_id",
            "date": "race_date"
        })
)


In [0]:
display(races_remaned_df)

#### Remove duplicate records 

In [0]:
races_distinct_df = races_remaned_df.dropDuplicates(["season","round"])

In [0]:
display(races_distinct_df)

#### Transform values of column race_name to Title Case 

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
        
)

In [0]:
display(races_final_df)

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table("silver_table"))